# NLP Masterclass 03: Recurrent Networks & LSTMs

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** NLP 02 (Embeddings), DL 01 (Neural Network Mathematics)  
**Difficulty:** ⭐⭐⭐ Intermediate/Advanced  
**Estimated Time:** 120–180 minutes

---

## 🎓 Socratic Deep Check

> *"Vanilla RNNs are logically sound for processing sequences, yet they practically fail for any dependency longer than ~10-20 steps. If the math for backpropagation (BPTT) is consistent, why does the gradient 'die' instead of flowing? Is it a hardware limitation or a fundamental mathematical property of recurrence?"*

---

## The Core Problem: Why Recurrence?

1. **Variable Length Input**: Traditional MLPs require fixed-size inputs. Natural language is dynamic—a sentence can be 3 words or 3,000.
2. **The Stationary Property**: The grammar rules that apply at the beginning of a sentence often apply at the end. We need **Shared Weights** to exploit this temporal invariance.
3. **Contextual Memory**: A word's meaning depends on its history. To understand "it" in "The cat ate the mouse and it died," you need to retain state from several steps back.

---

### 🎓 The Senior Perspective: Evolutionary Architecture

A Senior Engineer views RNN architectures as a bridge. They understand that while Transformers dominate large-scale NLP, RNNs introduced the fundamental concept of **distributed state** and **gating mechanisms** (LSTMs/GRUs) which paved the way for the internal logic of Attention blocks.

**Mental Model:** 
- **Vanilla RNN**: Reading word-by-word with a single, finite memory bucket. If the bucket overflows or leaks (vanishing gradient), you forget the beginning.
- **LSTM**: Reading with a notebook (Cell State) and an eraser (Forget Gate). You decide exactly what to keep and what to trash.
- **Transformer**: Having the whole book open on a table and "looking" at relevant words for every word produced (Search-and-Select model).

---

In [ ]:
!pip install -q torch numpy matplotlib tqdm

## 1 · Vanilla RNN: The Recurrence Formula

In a standard Feed-Forward network, each input $x$ is processed independently. In an RNN, we maintain a hidden state $h_t$ that carries information from the past ($h_{t-1}$).

$$
\begin{aligned}
h_t &= \phi(W_{hh} h_{t-1} + W_{xh} x_t + b_1) \\
y_t &= W_{hy} h_t + b_2
\end{aligned}
$$

Where $\phi$ is typically `tanh` or `ReLU`.

### Backpropagation Through Time (BPTT)

To compute the gradient of the loss $L$ with respect to the recurrent weights $W_{hh}$, we must use the chain rule across all previous time steps:

$$\frac{\partial L_t}{\partial W_{hh}} = \sum_{k=1}^t \frac{\partial L_t}{\partial h_t} \cdot \frac{\partial h_t}{\partial h_k} \cdot \frac{\partial h_k}{\partial W_{hh}}$$

The "middle term" $\frac{\partial h_t}{\partial h_k}$ is the culprit for the vanishing gradient problem.

### 🔬 Mathematical Proof: Vanishing Gradients

Let's look at the term $\frac{\partial h_t}{\partial h_k}$ for some $k < t$. By the chain rule:

$$\frac{\partial h_t}{\partial h_k} = \prod_{j=k+1}^t \frac{\partial h_j}{\partial h_{j-1}}$$

Each individual Jacobian $\frac{\partial h_j}{\partial h_{j-1}}$ is:
$$\frac{\partial h_j}{\partial h_{j-1}} = W_{hh}^T \cdot \text{diag}(\phi'(z_j))$$

Where $z_j = W_{hh} h_{j-1} + W_{xh} x_j + b_1$. If we take the norm:

$$\| \frac{\partial h_t}{\partial h_k} \| \le \prod_{j=k+1}^t \| W_{hh}^T \| \cdot \| \text{diag}(\phi'(z_j)) \|$$

**Crucial Insight:** 
1. For the `tanh` activation, the derivative $\phi'(z) \in (0, 1]$. 
2. If the largest eigenvalue (spectral radius) of $W_{hh}$ is less than 1, the product $\prod \| W_{hh} \|$ will shrink exponentially as $t-k$ increases.
3. Result: $\frac{\partial L_t}{\partial h_k} \to 0$ for large $t - k$. The model **cannot learn dependencies** that span beyond a few steps.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

def plot_gradient_flow(seq_len=50, w_scale=0.8):
    """Simulate the gradient flow backward through a vanilla RNN."""
    # Simplified: d_h(t)/d_h(t-1) = W * phi'(z)
    # We'll assume phi'(z) is constant for visualization (tanh max derivative is 1)
    
    # Random weight matrix with specific scale
    W = np.random.randn(64, 64) * w_scale / np.sqrt(64)
    
    norms = []
    grad = np.random.randn(64, 1) # Initial grad at step T
    
    for _ in range(seq_len):
        grad = W.T @ grad
        norms.append(np.linalg.norm(grad))
    
    return norms

plt.figure(figsize=(10, 5))
plt.plot(plot_gradient_flow(w_scale=0.5), label='||W|| = 0.5 (Vanishing)', color='#3498db', lw=2)
plt.plot(plot_gradient_flow(w_scale=1.1), label='||W|| = 1.1 (Exploding)', color='#e74c3c', lw=2)
plt.yscale('log')
plt.xlabel('Steps Backward in Time')
plt.ylabel('Gradient Norm')
plt.title('The Vanishing/Exploding Gradient Physics')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2 · LSTM: Long Short-Term Memory

LSTMs (Hochreiter & Schmidhuber, 1997) solve vanishing gradients by introducing a **Cell State** ($C_t$) that acts as a protected conveyor belt. Gradients can flow through $C_t$ for many steps without being repeatedly multiplied by $W_{hh}$, thanks to the **Forget Gate**.

### The Gating Mechanism

1. **Forget Gate** ($f_t$): $\sigma(W_f[h_{t-1}, x_t] + b_f)$ - Decides what to drop from memory.
2. **Input Gate** ($i_t$): $\sigma(W_i[h_{t-1}, x_t] + b_i)$ - Decides what new info to store.
3. **Candidate State** ($\tilde{C}_t$): $\tanh(W_C[h_{t-1}, x_t] + b_C)$ - The actual new info.
4. **Output Gate** ($o_t$): $\sigma(W_o[h_{t-1}, x_t] + b_o)$ - Decides what to reveal as the hidden state.

**State Updates:**
$$
\begin{aligned}
C_t &= f_t \odot C_{t-1} + i_t \odot \tilde{C}_t \\
h_t &= o_t \odot \tanh(C_t)
\end{aligned}
$$

The linear addition in $C_t$ update is the "gradient highway".

In [ ]:
class LSTMCell(nn.Module):
    """Elite Implementation: LSTM Cell from Scratch."""
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # Combine all gates into one linear layer (f, i, C_tilde, o)
        self.xh = nn.Linear(input_dim + hidden_dim, 4 * hidden_dim)
        
    def forward(self, x, states):
        (h_prev, c_prev) = states
        combined = torch.cat([x, h_prev], dim=1)
        
        gates = self.xh(combined)
        f, i, c_tilde, o = gates.chunk(4, dim=1)
        
        f = torch.sigmoid(f)
        i = torch.sigmoid(i)
        c_tilde = torch.tanh(c_tilde)
        o = torch.sigmoid(o)
        
        c_curr = f * c_prev + i * c_tilde
        h_curr = o * torch.tanh(c_curr)
        
        return h_curr, c_curr

print("LSTM Cell instantiated with gate chunking strategy for GPU efficiency.")

## 2b · GRU: Gated Recurrent Unit

GRUs (Cho et al., 2014) are a popular alternative to LSTMs. They are computationally cheaper because they have fewer gates (2 instead of 3) and merge the cell state into the hidden state.

**Equations:**
1. **Update Gate** ($z_t$): $\sigma(W_z[h_{t-1}, x_t])$
2. **Reset Gate** ($r_t$): $\sigma(W_r[h_{t-1}, x_t])$
3. **Candidate State** ($\tilde{h}_t$): $\tanh(W_h[r_t \odot h_{t-1}, x_t])$
4. **Hidden State**: $h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$

| Feature | LSTM | GRU |
|---------|------|-----|
| Gates | 3 | 2 |
| Parameters | Higher | Lower (~3/4) |
| Speed | Slower | Faster |
| Performance | Often slightly better on long seq | Often similar/better on short seq |

In [ ]:
class GRUCell(nn.Module):
    """Elite Implementation: GRU Cell from Scratch."""
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.update_gate = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.reset_gate = nn.Linear(input_dim + hidden_dim, hidden_dim)
        self.candidate = nn.Linear(input_dim + hidden_dim, hidden_dim)
        
    def forward(self, x, h_prev):
        combined = torch.cat([x, h_prev], dim=1)
        
        z = torch.sigmoid(self.update_gate(combined))
        r = torch.sigmoid(self.reset_gate(combined))
        
        combined_reset = torch.cat([x, r * h_prev], dim=1)
        h_tilde = torch.tanh(self.candidate(combined_reset))
        
        h_curr = (1 - z) * h_prev + z * h_tilde
        return h_curr

print("GRU Cell implemented. Note the 'Reset Gate' which allows the model to drop state entirely.")

## 3 · Bidirectional RNNs (BiLSTMs)

In many NLP tasks, future context is just as important as the past. Consider:
> "I went to the **bank** to fish."
> "I went to the **bank** to deposit money."

A unidirectional LSTM reading from left-to-right only sees "I went to the..." when it reaches "bank". It needs context to the right of the word to disambiguate.

**Architecture:** Two independent LSTMs (Forward & Backward) whose outputs are concatenated.
$$h_t = [\vec{h}_t ; \overleftarrow{h}_t]$$

**Note:** BiRNNs cannot be used for causal tasks (Autoregressive Lexical Generation) because the future is not yet known.

## 4 · Production Pipeline: POS Tagging with BiLSTM

Sequence tagging (assigning a label to *every* token) is the quintessential RNN task. We will implement it with **Packed Sequences** for optimal performance.

### Why Pack Sequences?
In a batch, sequences have variable lengths. Padding (filling with zeros) wastes computation. `pack_padded_sequence` tells PyTorch to skip the zeros during the recurrent steps, significantly speeding up training.

In [ ]:
import torch.optim as optim
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# 1. Synthetic Dataset
training_data = [
    ("The cat ate the mouse".split(), ["DET", "NN", "VB", "DET", "NN"]),
    ("A quick brown fox".split(), ["DET", "JJ", "JJ", "NN"]),
    ("Jumps over the lazy dog".split(), ["VB", "IN", "DET", "JJ", "NN"]),
    ("The clever professor taught the class".split(), ["DET", "JJ", "NN", "VB", "DET", "NN"])
]

word_to_ix = {"<PAD>": 0}
for sent, tags in training_data:
    for word in sent:
        if word not in word_to_ix: word_to_ix[word] = len(word_to_ix)

tag_to_ix = {"<PAD>": 0, "DET": 1, "NN": 2, "VB": 3, "JJ": 4, "IN": 5}

def prepare_sequence(seq, to_ix):
    idxs = [to_ix[w] for w in seq]
    return torch.tensor(idxs, dtype=torch.long)

print(f"Vocab size: {len(word_to_ix)}")

In [ ]:
class BiLSTMPOSTagger(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # The BiLSTM
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, 
                           num_layers=1, bidirectional=True, batch_first=True)
        
        # Classifier: hidden_dim * 2 because it's bidirectional
        self.hidden2tag = nn.Linear(hidden_dim * 2, tagset_size)

    def forward(self, sentence_padded, lengths):
        embeds = self.word_embeddings(sentence_padded)
        
        # PACKING: Skip computation on pads
        packed_embeds = pack_padded_sequence(embeds, lengths, 
                                            batch_first=True, enforce_sorted=False)
        
        lstm_out, _ = self.lstm(packed_embeds)
        
        # UNPACKING
        output, _ = pad_packed_sequence(lstm_out, batch_first=True)
        
        tag_space = self.hidden2tag(output)
        return tag_space

In [ ]:
EMBEDDING_DIM = 32
HIDDEN_DIM = 32

model = BiLSTMPOSTagger(len(word_to_ix), len(tag_to_ix), EMBEDDING_DIM, HIDDEN_DIM)
loss_function = nn.CrossEntropyLoss(ignore_index=0) # Ignore <PAD>
optimizer = optim.Adam(model.parameters(), lr=0.1)

# Prepare mini-batch (normally handled by DataLoader & Collator)
def get_batch():
    sents = [prepare_sequence(s, word_to_ix) for s, t in training_data]
    tags = [prepare_sequence(t, tag_to_ix) for s, t in training_data]
    lengths = [len(s) for s in sents]
    
    # Padding
    padded_sents = torch.nn.utils.rnn.pad_sequence(sents, batch_first=True, padding_value=0)
    padded_tags = torch.nn.utils.rnn.pad_sequence(tags, batch_first=True, padding_value=0)
    
    return padded_sents, padded_tags, lengths

print("Starting Training...")
for epoch in range(50):
    model.zero_grad()
    sents, tags, lengths = get_batch()
    
    tag_scores = model(sents, lengths)
    
    # Reshape for CrossEntropyLoss: (Batch * Seq, Classes)
    loss = loss_function(tag_scores.view(-1, len(tag_to_ix)), tags.view(-1))
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")

print("\n--- TEST IT ---")
with torch.no_grad():
    test_sent, _, test_len = get_batch()
    scores = model(test_sent, test_len)
    predictions = torch.argmax(scores, dim=2)
    print(f"Example prediction for 'The cat ate the mouse':")
    predicted_tags = [list(tag_to_ix.keys())[idx] for idx in predictions[0] if idx != 0]
    print(predicted_tags)

## 5 · Performance & The Evolution to Attention

### The Serial Bottleneck
While LSTMs and GRUs fixed the **vanishing gradient**, they did not fix the **computational efficiency** problem.

Modern GPUs have thousands of cores. 
- **CNNs** are fully parallelable across space.
- **Transformers** are fully parallelable across time.
- **RNNs** are **strictly sequential**. You cannot compute $h_t$ until $h_{t-1}$ is finished. This means TFLOPS of GPU power sit idle as the model processes one word at a time.

---

## ✅ Knowledge Check

### Q1: The Forget Gate
If you initialize an LSTM's forget gate biases to a high positive value (e.g., +1.0 or +2.0) during training, what problem are you trying to mitigate?

<details><summary>👉 Answer</summary>
You are mitigating the vanishing gradient. By setting $b_f$ high, $f_t \approx 1$ initially, which forces the model to keep most of the memory in the early stages of training, allowing gradients to flow further back.
</details>

### Q2: Bidirectionality
Can you use a BiLSTM for a real-time speech-to-text system where latency must be < 100ms?

<details><summary>👉 Answer</summary>
Technically no (or with great difficulty). A BiLSTM requires the **entire** sequence to be finished before it can compute the backward hidden states. In real-time systems, you haven't heard the end of the sentence yet. You would need to use lookahead-windows or strictly unidirectional models.
</details>

### Q3: GRU vs LSTM
Which architecture is generally more prone to exploding gradients (without clipping)?

<details><summary>👉 Answer</summary>
Both are safer than Vanilla RNNs, but because they both rely on gating which can produce large activations if not normalized. However, the exploding gradient is usually solved by **Gradient Clipping**, which is non-negotiable for RNN training.
</details>

---

**Next →** [NLP 04: Sequence-to-Sequence & Attention Mechanisms]()